In [189]:
# Import all the libraries used in this notebook

import pandas as pd
from glob import glob
from os.path import join
import numpy as np
from pvlib.solarposition import get_solarposition
import datetime
import os 
print(os.environ['CONDA_DEFAULT_ENV']) # Check the name of the current Conda environment

### Helper Functions

In [246]:
def drop_variable(df, *args): # delete a variable from the dataframe
    for arg in args:
        df = df.drop(arg, axis=1)

    return df

def dropna_in_variable(df, *args): # delete the rows where a nan exists in the specified column(s)
    df = df.dropna(subset=(args))
    return df

## MVCO Raw file (before processing)

In [191]:
mvco_raw= pd.read_csv('qced_MVCO_ocn_sonic_vaisala_QC_all_data.2004-2023.csv')

In [209]:
a = 'sdfsd'
b = 'sdf'
c = 2432423423423

print(f"{a:30s} {b:20s}: {c:.15f}")

sdfsd                          sdf                 : 2432423423423.000000000000000


In [214]:
print("Total datapoints/timestamps (20 min intervals): {0}".format(len(mvco_raw)))
print("Total missing values of - {0}:{1:8}".format('wave_period:0_m:deg', mvco_raw['wave_period:0_m:deg'].isna().sum()))
print("Total present values of - {0}:{1:8}".format('wave_period:0_m:deg', len(mvco_raw) - mvco_raw['wave_period:0_m:deg'].isna().sum()))

Total datapoints/timestamps (20 min intervals): 131176
Total missing values of - wave_period:0_m:deg:       0
Total present values of - wave_period:0_m:deg:  131176


In [204]:
# Set pandas to not cut out the middle of df in the display results
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)     # Show all rows (you can set it to a specific number if you want)

print("Total datapoints/timestamps (20 min intervals): {0} \n".format(len(mvco_raw)))
print("Column Name                      Percentage of Missing data\n")
print(mvco_raw.isna().sum()/len(mvco_raw)) # shows the percentage of data missing in the corresponding column/variable

Total datapoints/timestamps (20 min intervals): 131176 

Column Name                      Percentage of Missing data

DateTime                                  0.000000
wave_height:0_m:m                         0.000000
wave_period:0_m:deg                       0.000000
wave_dir:0_m:deg                          0.001410
water_temp:0_m:C                          0.000000
bottom_current:0_m:cm/s                   0.000000
bottom_current_dir:0_m:deg_toward         0.000000
near_surf_current:0_m:cm/s                0.001517
near_surf_current_dir:0_m:deg             0.001517
press-air_pres-nominal_depth:0_m:m        0.101025
water_temp_2:0_m:C                        0.178127
num_records_per_period:12_m:count         0.000000
air_temp:12_m:C                           0.006922
RH:12_m:%                                 0.000000
press:12_m:mb                             0.000000
air_temp_median:12_m:C                    0.006922
RH_median:12_m:%                          0.000000
press_median:12

In [236]:
print(mvco_raw.columns)

Index(['DateTime', 'wave_height:0_m:m', 'wave_period:0_m:deg',
       'wave_dir:0_m:deg', 'water_temp:0_m:C', 'bottom_current:0_m:cm/s',
       'bottom_current_dir:0_m:deg_toward', 'near_surf_current:0_m:cm/s',
       'near_surf_current_dir:0_m:deg', 'press-air_pres-nominal_depth:0_m:m',
       'water_temp_2:0_m:C', 'num_records_per_period:12_m:count',
       'air_temp:12_m:C', 'RH:12_m:%', 'press:12_m:mb',
       'air_temp_median:12_m:C', 'RH_median:12_m:%', 'press_median:12_m:mb',
       'air_temp_std:12_m:C', 'RH_std:12_m:%', 'press_std:12_m:mb',
       'air_temp_len:12_m:count', 'RH_len:12_m:count', 'pres_len:12_m:count',
       'num_records_per_period:18.4_m:count', 'wspd_3D1:18.4_m:m/s',
       'wdir_3D1:18.4_m:deg', 'w:18.4_m:m/s',
       'air_temp_speed_of_sound_3D1:18.4_m:C', 'wspd_3D1_inst:18.4_m:m/s',
       'wspd_3D1_inst_2:18.4_m:m/s', 'air_temp_speed_of_sound_3D1_2:18.4_m:C',
       'sig_U:18.4_m:UNK', 'sig_V:18.4_m:UNK', 'sig_W:18.4_m:UNK',
       'sig_T:18.4_m:UNK', 'uv

In [251]:
#                                   replace these variable names with the ones you want to drop
mvco_raw = drop_variable(mvco_raw, 'air_temp_len:12_m:count', 'RH_QC', 'P_QC', 'Ha_QC')

In [ ]:
#                                   replace these variable names with the ones you want to clean
mvco_raw = dropna_in_variable(mvco_raw, 'air_temp_len:12_m:count', 'RH_QC', 'P_QC', 'Ha_QC')

In [252]:
print(mvco_raw.columns)

Index(['DateTime', 'wave_height:0_m:m', 'wave_period:0_m:deg',
       'wave_dir:0_m:deg', 'water_temp:0_m:C', 'bottom_current:0_m:cm/s',
       'bottom_current_dir:0_m:deg_toward', 'near_surf_current:0_m:cm/s',
       'near_surf_current_dir:0_m:deg', 'press-air_pres-nominal_depth:0_m:m',
       'water_temp_2:0_m:C', 'num_records_per_period:12_m:count',
       'air_temp:12_m:C', 'RH:12_m:%', 'press:12_m:mb',
       'air_temp_median:12_m:C', 'RH_median:12_m:%', 'press_median:12_m:mb',
       'air_temp_std:12_m:C', 'RH_std:12_m:%', 'press_std:12_m:mb'],
      dtype='object')


In [257]:
def count_no_missing(df, *args):
    # Count rows where there is no missing data in all specified columns
    no_missing_data_count = df[args].notna().all(axis=1).sum()

    # Display the count of rows with no missing data in all specified columns
    print("Count of rows with no missing data in all specified columns: {0}".format(no_missing_data_count))
    print("Num Hours: {0}".format(no_missing_data_count / 3))
    print("Num Days: {0}".format(no_missing_data_count / 3 / 24))
    print("Num Weeks: {0}".format(no_missing_data_count / 3 / 24 / 7))
    print("Num Months: {0}".format(no_missing_data_count / 3 / 24 / 7 / 4))
    print("Num Years: {0}".format(no_missing_data_count / 3 / 24 / 7 / 4 / 12))
    print("Num Years: {0}".format(no_missing_data_count / 3 / 24 / 365))

In [258]:
count_no_missing(mvco_raw, "wave_height:0_m:m", "P_QC", "RH_QC", "AT_QC", "SST_QC", "u1_QC", "v1_QC")

KeyError: ('wave_height:0_m:m', 'P_QC', 'RH_QC', 'AT_QC', 'SST_QC', 'u1_QC', 'v1_QC')

In [197]:

mvco_raw = mvco_raw.drop('salinity:0_m:PSU', axis=1)

## MVCO file post-processed

In [ ]:
# mvco= pd.read_csv('mvco_mlsl_qc.csv')
mvco= pd.read_csv('mvco_mlsl_qc.csv', encoding='utf-8', delimiter=',')
# print(mvco.columns)

In [ ]:
# Convert the data to time series data with date time as the index
mvco['Time']= pd.to_datetime(mvco['Time'] )
mvco.index = mvco['Time'] 

In [ ]:
# List the data variables which we have
# print(len(mvco.columns))
# for i in mvco.columns:
#     print(i)

In [ ]:
print(len(mvco))
print(mvco.isna().sum()/len(mvco))

In [ ]:

# Count rows where there is no missing data in all specified columns
no_missing_data_count = mvco[["potential_temperature:12_m:K",
                             "mixing_ratio:12_m:g_kg-1",
                             "skin_virtual_potential_temperature:0_m:K",
                             "wind_speed:12_m:m_s-1"]].notna().all(axis=1).sum()

# Display the count of rows with no missing data in all specified columns
print("\nCount of rows with no missing data in all specified columns:")
print(no_missing_data_count)

In [ ]:

# Count rows where there is no missing data in all specified columns
no_missing_data_count = mvco[['wave_period:0_m:s',
                              "wave_phase_speed:0_m:m_s-1",
                              'u_wave:0_m:m_s-1',
                              'v_wave:0_m:m_s-1',
                              "u_wind:18.4_m:m_s-1",
                              "v_wind:18.4_m:m_s-1",
                            #   'angle_between_wind_wave:0_m:degrees'
                              ]].notna().all(axis=1).sum()

# Display the count of rows with no missing data in all specified columns
print("\nCount of rows with no missing data in all specified columns:")
print(no_missing_data_count)

In [ ]:
nn = len(mvco) - mvco['bulk_richardson:18.4_m:none'].isna().sum()
print(nn)
print(nn / 3 / 24)
print(nn / 3 / 24 / 365)

In [ ]:
nn = mvco['angle_between_wind_wave:0_m:degrees'].isna().sum()
print(nn)
all = len(mvco)
print(all - nn)
print((all - nn) / all * 100)

In [ ]:
print(len(mvco) - mvco['surface_roughness_drennan:0_m:m'].isna().sum())
nn = len(mvco) - mvco['surface_roughness_drennan:0_m:m'].isna().sum()
print(nn / 3 / 24)

In [ ]:
print(mvco['wave_height:0_m:m'].notna().sum())
nn = mvco['wave_height:0_m:m'].isna().sum()
print(nn / 3 / 24 / 365)

In [ ]:
print(mvco['wind_speed:18.4_m:m_s-1'].notna().sum())
nn = mvco['wind_speed:18.4_m:m_s-1'].isna().sum()
print(nn / 3 / 24 / 365)

In [ ]:
print(len(mvco) - mvco['wind_speed:12_m:m_s-1'].isna().sum())
nn = len(mvco) - mvco['wind_speed:12_m:m_s-1'].isna().sum()
print(nn / 3 / 24)

In [ ]:
# these seem to be missing ( we know from plotting below, confirming that all values are NaN)

#
# total number of data samples
#
print(len(mvco.index))

#
# Number of samples with Nan for specific columns
#
print(mvco["temperature_med:12_m:K"].isna().sum())
print(mvco["pressure_med:12_m:hPa"].isna().sum())
print(mvco['wind_direction_inst:18.4_m:degrees'].isna().sum())

## for graphing

In [ ]:
import math
#
# Visualizing time series data 
# Run 'pip install plotly'
# Restart jupyter with increased data rate option since the dataset is so large
# 'jupyter notebook --NotebookApp.iopub_data_rate_limit=1.0e10'
#
import plotly
import chart_studio.plotly as py
from plotly.graph_objs import Scatter, Layout
import plotly.graph_objs as go

from  plotly.graph_objs import *


In [ ]:

#
# Define this time series layout once, will use again in time series plots in rest of notebook 
#
timeSeriesLayout = dict(
    title='Time Series with Rangeslider',
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count =1,
                    label='1d',
                    step='day',
                    stepmode='forward'),
                dict(count=7,
                     label='1w',
                     step='week',
                     stepmode='forward'),
                dict(count=30,
                     label='1m',
                     step='month',
                     stepmode='forward'),
                dict(step='all')
            ])
        ),
        rangeslider=dict(),
        type='date'
    ),
    width=1200,  # Custom width in pixels
    height=800,  # Custom height in pixels
)


layout = timeSeriesLayout

names of the variables for each trace

- trace1 = azimuth
- trace2 = zenith
- trace3 = temp12m_K
- trace4 = N/A
- trace5 = std temp 12m
- trace6 = temp18.4m_K
- trace7 = tmp2 18.4
- trace8 = water surface temp
- trace9 = pressure12m
- trace9b = pressure18.4m
- trace10 = N/A
- trace11 = std pres 12m
- trace12 = pot temp 12m
- trace12b = pot temp 18.4m
- trace13 = Skin virtual potential temp 0m
- trace14 = mixing ratio 0m
- trace15 = mixing ratio 12m
- trace15b = mixing ratio 18.4m
- trace16 = rh 12m
- trace17 = wave height 0m
- trace18 = wave period 0m
- trace19 = wave phase speed 0m
- trace20 = near surf current speed 0m
- trace21 = wave dir degrees 0m
- trace22 = surface current dir 0m
- trace23 = wind dir 18.4 m
- trace23b = angle between wind wave 0m
- trace23c = u_wave:0_m:m_s-1
- trace24 = uw cov 18.4m
- trace25 = vw cov 18.4m
- trace26 = friction vel 18.4m
- trace27 = kinematic_sensible_heat_flux:18.4_m
- trace28 = temp scale 18.4m
- trace29 = momentum_flux_18.4m
- trace30 = heat_flux_18.4m
- trace31 = wind_speed:12_m:m_s-1
- trace32 = wind_speed:18.4_m:m_s-1
- trace33 = bulk_richardson_12m
- trace33b = bulk_richardson_18.4m
- trace34 = surface_roughness_drennan_0m

In [177]:
trace1 = go.Scatter(
    x =  mvco.index,
    y =  mvco['azimuth:0_m:degrees'],
    name = "azimuth",
    line = dict(color = 'blue'),
    connectgaps = False,
    opacity = 0.5)
trace2 = go.Scatter(
    x =  mvco.index,
    y =  mvco['zenith:0_m:degrees'],
    name = "zenith",
    line = dict(color = 'green'),
    connectgaps = False,
    opacity = 0.5)
trace3 = go.Scatter(
    x =  mvco.index,
    y =  mvco['temperature:12_m:K'],
    name = "temp12m",
    line = dict(color = 'green'),
    connectgaps = False,
    opacity = 0.5)

trace5 = go.Scatter(
    x =  mvco.index,
    y =  mvco['temperature_std:12_m:K'],
    name = "std temp 12m",
    line = dict(color = 'cyan'),
    connectgaps = False,
    opacity = 0.5)
trace6 = go.Scatter(
    x =  mvco.index,
    y =  mvco['temperature:18.4_m:K'],
    name = "temp18.4m",
    line = dict(color = 'blue'),
    connectgaps = False,
    opacity = 0.5)
trace7 = go.Scatter(
    x =  mvco.index,
    y =  mvco['temperature2:18.4_m:K'],
    name = "tmp2 18.4",
    line = dict(color = 'purple'),
    connectgaps = False,
    opacity = 0.5)
trace8 = go.Scatter(
    x =  mvco.index,
    y =  mvco['water_sfc_temperature:0_m:K'],
    name = "water surface temp",
    line = dict(color = 'red'),
    connectgaps = False,
    opacity = 0.5)
trace9 = go.Scatter(
    x =  mvco.index,
    y =  mvco['pressure:12_m:hPa'],
    name = "pressure12m",
    line = dict(color = 'green'),
    connectgaps = False,
    opacity = 0.5)
trace9b = go.Scatter(
    x =  mvco.index,
    y =  mvco['pressure:18.4_m:hPa'],
    name = "pressure18.4m",
    line = dict(color = 'green'),
    connectgaps = False,
    opacity = 0.5)

trace11 = go.Scatter(
    x =  mvco.index,
    y =  mvco['pressure_std:12_m:hPa'],
    name = "std pres 12m",
    line = dict(color = 'cyan'),
    connectgaps = False,
    opacity = 0.5)
trace12 = go.Scatter(
    x =  mvco.index,
    y =  mvco['potential_temperature:12_m:K'],
    name = "pot temp 12m",
    line = dict(color = 'blue'),
    connectgaps = False,
    opacity = 0.5)
trace12b = go.Scatter(
    x =  mvco.index,
    y =  mvco['potential_temperature:18.4_m:K'],
    name = "pot temp 18.4m",
    line = dict(color = 'blue'),
    connectgaps = False,
    opacity = 0.5)
trace13 = go.Scatter(
    x =  mvco.index,
    y =  mvco['skin_virtual_potential_temperature:0_m:K'],
    name = "Skin virtual potential temp",
    line = dict(color = 'purple'),
    connectgaps = False,
    opacity = 0.5)
trace14 = go.Scatter(
    x =  mvco.index,
    y =  mvco['mixing_ratio:0_m:g_kg-1'],
    name = "mixing ratio 0m",
    line = dict(color = 'green'),
    connectgaps = False,
    opacity = 0.5)
trace15 = go.Scatter(
    x =  mvco.index,
    y =  mvco['mixing_ratio:12_m:g_kg-1'],
    name = "mixing ratio 12m",
    line = dict(color = 'gold'),
    connectgaps = False,
    opacity = 0.5)
trace15b = go.Scatter(
    x =  mvco.index,
    y =  mvco['mixing_ratio:18.4_m:g_kg-1'],
    name = "mixing ratio 18.4m",
    line = dict(color = 'gold'),
    connectgaps = False,
    opacity = 0.5)
trace16 = go.Scatter(
    x =  mvco.index,
    y =  mvco['relative_humidity:12_m:%'],
    name = "rh 12m",
    line = dict(color = 'blue'),
    connectgaps = False,
    opacity = 0.5)
trace16b = go.Scatter(
    x =  mvco.index,
    y =  mvco['relative_humidity:18.4_m:%'],
    name = "rh 18.4m",
    line = dict(color = 'blue'),
    connectgaps = False,
    opacity = 0.5)
trace17 = go.Scatter(
    x =  mvco.index,
    y =  mvco['wave_height:0_m:m'],
    name = "wave height",
    line = dict(color = 'blue'),
    connectgaps = False,
    opacity = 0.5)
trace18 = go.Scatter(
    x =  mvco.index,
    y =  mvco['wave_period:0_m:s'],
    name = "wave period",
    line = dict(color = 'cyan'),
    connectgaps = False,
    opacity = 0.5)
trace19 = go.Scatter(
    x =  mvco.index,
    y =  mvco['wave_phase_speed:0_m:m_s-1'],
    name = "wave phase speed",
    line = dict(color = 'green'),
    connectgaps = False,
    opacity = 0.5)
trace20 = go.Scatter(
    x =  mvco.index,
    y =  mvco['near_surf_current:0_m:m_s-1'],
    name ='near surf current speed',
    line = dict(color = 'gold'),
    connectgaps = False,
    opacity = 0.5)
trace21 = go.Scatter(
    x =  mvco.index,
    y =  mvco['wave_direction:0_m:degrees'],
    name = "wave dir degrees",
    line = dict(color = 'blue'),
    connectgaps = False,
    opacity = 0.5)
trace22 = go.Scatter(
    x =  mvco.index,
    y =  mvco['near_surf_current_dir:0_m:deg'],
    name = "surface current dir",
    line = dict(color = 'red'),
    connectgaps = False,
    opacity = 0.5)
trace23 = go.Scatter(
    x =  mvco.index,
    y =  mvco['wind_direction:18.4_m:degrees'],
    name = "wind dir 18.4 m",
    line = dict(color = 'gold'),
    connectgaps = False,
    opacity = 0.5)
trace23b = go.Scatter(
    x =  mvco.index,
    y =  mvco['angle_between_wind_wave:0_m:degrees'],
    name = "angle between wind wave",
    line = dict(color = 'green'),
    connectgaps = False,
    opacity = 0.5)
trace23c= go.Scatter(
    x =  mvco.index,
    y =  mvco["u_wave:0_m:m_s-1"],
    name = "u",
    line = dict(color = 'green'),
    connectgaps = False,
    opacity = 0.5)
trace24 = go.Scatter(
    x =  mvco.index,
    y =  mvco['u_w:18.4_m:m2_s-2'],
    name = "uw cov 18.4m",
    line = dict(color = 'blue'),
    connectgaps = False,
    opacity = 0.5)
trace25 = go.Scatter(
    x =  mvco.index,
    y =  mvco["v_w:18.4_m:m2_s-2"],
    name = "vw cov 18.4m",
    line = dict(color = 'green'),
    connectgaps = False,
    opacity = 0.5)
trace26 = go.Scatter(
    x =  mvco.index,
    y =  mvco['friction_velocity:18.4_m:m_s-1'],
    name = "friction vel 18.4m",
    line = dict(color = 'red'),
    connectgaps = False,
    opacity = 0.5)
trace27 = go.Scatter(
    x =  mvco.index,
    y =  mvco['kinematic_sensible_heat_flux:18.4_m:K_m_s-1'],
    name = "kinematic_sensible_heat_flux:18.4_m",
    line = dict(color = 'blue'),
    connectgaps = False,
    opacity = 0.5)
trace28 = go.Scatter(
    x =  mvco.index,
    y =  mvco["temperature_scale:18.4_m:K"],
    name = "temp scale 18.4m",
    line = dict(color = 'green'),
    connectgaps = False,
    opacity = 0.5)

trace29 = go.Scatter(
    x =  mvco.index,
    y =  mvco['momentum_flux:18.4_m:m2_s-2'],
    name = "momentum_flux:18.4_m:m2_s-2",
    line = dict(color = 'blue'),
    connectgaps = False,
    opacity = 0.5)
trace30 = go.Scatter(
    x =  mvco.index,
    y =  mvco["heat_flux:18.4_m:degrees_C_m_s-1"],
    name = "heat_flux:18.4_m:degrees_C_m_s-1",
    line = dict(color = 'green'),
    connectgaps = False,
    opacity = 0.5)

trace31 = go.Scatter(
    x =  mvco.index,
    y =  mvco["wind_speed:12_m:m_s-1"],
    name = "wind_speed:12_m:m_s-1",
    line = dict(color = 'red'),
    connectgaps = False,
    opacity = 0.5)
trace32 = go.Scatter(
    x =  mvco.index,
    y =  mvco["wind_speed:18.4_m:m_s-1"],
    name = "wind_speed:18.4_m:m_s-1",
    line = dict(color = 'green'),
    connectgaps = False,
    opacity = 0.5)

trace33 = go.Scatter(
    x =  mvco.index,
    y =  mvco['bulk_richardson:12_m:none'],
    name = 'bulk_richardson:12_m:none',
    line = dict(color = 'orange'),
    connectgaps = False,
    opacity = 0.5)
trace33b = go.Scatter(
    x =  mvco.index,
    y =  mvco['bulk_richardson:18.4_m:none'],
    name = 'bulk_richardson:18.4_m:none',
    line = dict(color = 'orange'),
    connectgaps = False,
    opacity = 0.5)
trace34 = go.Scatter(
    x =  mvco.index,
    y =  mvco['surface_roughness_drennan:0_m:m'],
    name = 'surface_roughness_drennan:0_m:m',
    line = dict(color = 'purple'),
    connectgaps = False,
    opacity = 0.5)


In [ ]:
nn = mvco['L'].isna().sum()
print(nn)
print('all', all / 3 / 24 / 365)
print(nn / 3 / 24 / 365)

all = len(mvco)
print((all - nn) / 3 / 24 / 365)
# print((all - nn) / all * 100)

In [ ]:
def drawTraces(*args):
    data = [*args]
    plotly.offline.iplot(data, filename='basic-line-plot')
    layout = timeSeriesLayout

In [187]:
inputs = ["Time",
          "zenith:0_m:degrees",
          "azimuth:0_m:degrees",
          "water_sfc_temperature:0_m:K",
          "pressure:18.4_m:hPa",
          "mixing_ratio:18.4_m:g_kg-1",
          "relative_humidity:18.4_m:%",
          "wave_direction:0_m:degrees",
          "wave_height:0_m:m",
          "wave_period:0_m:s",
          "angle_between_wind_wave:0_m:degrees",
          "wind_speed:18.4_m:m_s-1",
          "wind_direction:18.4_m:degrees",
          "bulk_richardson:18.4_m:none"]


In [ ]:

inputTraces = [trace1, trace2, trace8, trace9b, trace15b, trace16b, trace21, trace17, trace18, trace23b, trace32, trace23, trace33b]

drawTraces(*inputTraces)

In [188]:
mvco_split = mvco[inputs]

print(mvco_split.columns )

# mvco_split = mvco_split.drop(inputs, axis=1)

# print(mvco_split.columns )

Index(['Time', 'zenith:0_m:degrees', 'azimuth:0_m:degrees',
       'water_sfc_temperature:0_m:K', 'pressure:18.4_m:hPa',
       'mixing_ratio:18.4_m:g_kg-1', 'relative_humidity:18.4_m:%',
       'wave_direction:0_m:degrees', 'wave_height:0_m:m', 'wave_period:0_m:s',
       'angle_between_wind_wave:0_m:degrees', 'wind_speed:18.4_m:m_s-1',
       'wind_direction:18.4_m:degrees', 'bulk_richardson:18.4_m:none'],
      dtype='object')


### the rest of graphing

In [ ]:
#
# Missing
#
#trace10 = go.Scatter(
#    x =  mvco.index,
#    y =  mvco['pressure_med:12_m:hPa'],
#    name = "median press 12m",
#    line = dict(color = 'gold'),
#    connectgaps = False,
#    opacity = 0.5)

In [ ]:
#kinematic_sensible_heat_flux:18.4_m:K_m_s-1
#temperature_scale:18.4_m:K
# 
# this is missing
#trace4 = go.Scatter(
#    x =  mvco.index,
#    y =  mvco['temperature_med:12_m:K'],
#    name = "median temp 12m",
#    line = dict(color = 'gold'),
#    connectgaps = False,
#    opacity = 0.5)
#

### different graphs

In [ ]:
# data = [trace29,trace30, trace17]
data = [trace23b]
plotly.offline.iplot(data, filename='basic-line-plot')
layout = timeSeriesLayout

In [ ]:
data = [trace29,trace30, trace17]
# data = [trace23b]
plotly.offline.iplot(data, filename='basic-line-plot')
layout = timeSeriesLayout

In [ ]:
data = [ trace1, trace2]
plotly.offline.iplot(data, filename='basic-line-plot')

In [ ]:
# data = [trace12,trace15,trace13,trace31, trace32]
data = [trace31, trace32, trace17, trace19, trace33b]
plotly.offline.iplot(data, filename='basic-line-plot')
layout = timeSeriesLayout

In [ ]:
data = [trace34]
plotly.offline.iplot(data, filename='basic-line-plot')
layout = timeSeriesLayout

In [ ]:
data = [trace31, trace17, trace26, trace34]
plotly.offline.iplot(data, filename='basic-line-plot')
layout = timeSeriesLayout

In [ ]:
data = [trace12b,trace15b,trace13,trace31, trace32]
plotly.offline.iplot(data, filename='basic-line-plot')
layout = timeSeriesLayout

In [ ]:
data = [ trace3,trace5,trace6,trace7, trace8]
plotly.offline.iplot(data, filename='basic-line-plot')
layout = timeSeriesLayout

In [ ]:
data = [ trace9,trace11]
plotly.offline.iplot(data, filename='basic-line-plot')
layout = timeSeriesLayout

In [ ]:
data = [trace12,trace13]
plotly.offline.iplot(data, filename='basic-line-plot')
layout = timeSeriesLayout

In [ ]:
data = [trace14,trace15]
plotly.offline.iplot(data, filename='basic-line-plot')
layout = timeSeriesLayout

In [ ]:
data = [trace16]
plotly.offline.iplot(data, filename='basic-line-plot')
layout = timeSeriesLayout

In [ ]:
data = [trace17,trace18, trace19, trace20]
plotly.offline.iplot(data, filename='basic-line-plot')
layout = timeSeriesLayout

In [ ]:
data = [trace21,trace22, trace23, trace23b]
plotly.offline.iplot(data, filename='basic-line-plot')
layout = timeSeriesLayout

In [ ]:

data = [trace24,trace25, trace26]
plotly.offline.iplot(data, filename='basic-line-plot')
layout = timeSeriesLayout

In [ ]:

data = [trace27,trace28]
plotly.offline.iplot(data, filename='basic-line-plot')
layout = timeSeriesLayout


# data = [trace12,trace15,trace13,trace31, trace32]
data = [trace31, trace32, trace17, trace19]
plotly.offline.iplot(data, filename='basic-line-plot')
layout = timeSeriesLayout

In [ ]:
#
# Determine the distribution of stable and unstable cases
#
myNumList = np.arange(-1.0,1,.01)

# bulk_richardson = "bulk_richardson:12_m:none"
bulk_richardson = "bulk_richardson:18.4_m:none"

#print (myNumList)
mvco.hist(column = bulk_richardson,   bins = list(myNumList))

print("Unstable cases:" , mvco[bulk_richardson].loc[mvco[bulk_richardson]< -.02].count())
print("Stable cases: ", mvco[bulk_richardson].loc[mvco[bulk_richardson]> 0.02].count())
print("Neutral cases: ", mvco[bulk_richardson].loc[(mvco[bulk_richardson]>= -0.02) & (mvco[bulk_richardson]<= 0.02)].count())



In [ ]:
from matplotlib import pyplot as plt
#
# Determine the distribution of stable and unstable cases
#
print("Unstable cases:" , mvco[bulk_richardson].loc[mvco[bulk_richardson]< -.02].count())
print("Stable cases: ", mvco[bulk_richardson].loc[mvco[bulk_richardson]> 0.02].count())
print("Neutral cases: ", mvco[bulk_richardson].loc[(mvco[bulk_richardson]>= -0.02) & (mvco[bulk_richardson]<= 0.02)].count())

#n, bins, patches = plt.hist(mvcoRaw[bulk_richardson], bins = [mvcoRaw[bulk_richardson].min(), -.02,.02, mvcoRaw[bulk_richardson].max()])
n, bins, patches = plt.hist(mvco[bulk_richardson], bins = [-.5, -.02,.02, .5])
patches[1].set_fc('r')
patches[0].set_fc('b')
patches[2].set_fc('g')

In [ ]:
import math
import numpy as np
from scipy.optimize import *
import numpy as np 
from mlsurfacelayer.mo import psi_h_branko
from mlsurfacelayer.mo import psi_m_branko
from mlsurfacelayer.mo import psi_h_alternate
from mlsurfacelayer.mo import psi_m_alternate

from mlsurfacelayer.mo import mo_similarity_offshore_branko
from mlsurfacelayer.mo import mo_similarity_offshore_alternate

In [ ]:
print(mvco.iloc[111])
print(mvco.iloc[112])
print(mvco.iloc[113])


In [ ]:
from math import log
# 
#
# the next section of code solves the MOST system of equations above for friction velocity and L, Obhukov length
# Windspeed at 18.4m other vars at Ri and potential temp at 12m
#

#
# Von Karman constant
#
k = .4

#
# The acceleration of gravity
# 
g = 9.8

#
# We will count the number of systems of equations in which bulk richardson number is < 0 and > 0 to keep 
# track of the convergence of systems representing unstable (Ri < 0) and stable ( Ri > 0) regimes.
#
riNegCount = 0
riNegConv = 0
riPosCount = 0
riPosConv = 0

#
# These are the solutions to our systems 
#
mvco["L-branko"] = np.nan
mvco["MOSTustar-branko"] = np.nan
mvco["L-alternate"] = np.nan
mvco["MOSTustar-alternate"] = np.nan

#
# From experience, stable regime is sensitive to the "first guess" of the root solver
# Keep track of the value of Ri in these cases. Maybe we can learn something
#
posNoConvRi = []

#
# Loop through the derived data samples, solve the system of equations as outlined above,
# record some statistics as well as the solutions
#
print(len(mvco.index))
for method in range(2):
    if method == 0:
        psi_h = psi_h_branko
        psi_m = psi_m_branko
        mo_tag = '-branko'
    elif method == 1:
        psi_h = psi_h_alternate
        psi_m = psi_m_alternate
        mo_tag = '-alternate'
        
    for i in range(0,len(mvco.index)):
        # print(i)
        
        #
        # Get the known variables for this instance
        #
        Ri = mvco[bulk_richardson].iloc[i]
        skinPotT = mvco['skin_virtual_potential_temperature:0_m:K'].iloc[i]
        T = mvco['water_sfc_temperature:0_m:K'].iloc[i] # check this
        wspd = mvco['wind_speed:18.4_m:m_s-1'].iloc[i]
        waveHt = mvco['wave_height:0_m:m'].iloc[i]
        # potT12 = mvco['potential_temperature:12_m:K'].iloc[i]
        potT18 = mvco['potential_temperature:18.4_m:K'].iloc[i]
        Cp = mvco['wave_phase_speed:0_m:m_s-1'].iloc[i]
        


        # Get column indices
        mostustar_col_index = mvco.columns.get_loc("MOSTustar" + mo_tag)
        l_col_index = mvco.columns.get_loc("L" + mo_tag)

        if math.isnan(Ri) or math.isnan(skinPotT) or math.isnan(T) or math.isnan(wspd) or math.isnan(waveHt) or math.isnan(potT18) or math.isnan(Cp):
            # mvco["MOSTustar"].iloc[i] = np.nan
            # mvco["L"].iloc[i] = np.nan
            mvco.iloc[i, mostustar_col_index] = np.nan
            mvco.iloc[i, l_col_index] = np.nan
            continue    
        #print("Ri: ", Ri)
        #print("skinPotT: ", skinPotT)
        #print("T: ", T)
        #print("wspd40: ", wspd40)
        #print("waveHt: ", waveHt)
        #print("potT40: ", potT40)
        # print("Cp: ", Cp )
        
        #
        # Define the system of equations for which we will find the roots
        #
        def myF(z):
            ustar = z[0]
            L = z[1]
            
            #
            # define surface roughness as a functio of friction velocity, wave height , and wave phase speed
            #
            z0 = 3.35 * waveHt * (ustar/Cp)**3.4
            
            #
            # Initalize the function outputs
            #
            F = np.empty((2))
            
            #
            # Theta(Z40) equation set equal to zero
            #
            F[0] = skinPotT +  ustar/((g/T)*k*L) *(log(12/z0)- psi_h(12,L,Ri) + psi_h(z0,L,Ri)) - potT18
            
            #
            # U(Z12) equation set equal to zero
            #
            F[1] = ustar/k * (log(12/z0) - psi_m(12,L,Ri) + psi_m(z0,L,Ri)) - wspd
            
            return F
        
        #
        # Define a first guess, [u*,L], to the system of equations depending on the stability regime
        #
        if Ri < 0:
            #
            # Experience suggests that these unstable systems are not so sensitive to first guess
            #
            zGuess = np.array([.1, -1.0])
        else:
            #
            # Experience suggests that these stable systems are very sensitive to first guess
            # More work needed here to find the optimal static guess that works for all or dependent on 
            # other inputs
            #
            zGuess = np.array([.1, 100])
            
        #
        # Solve the system if possible
        #
        z , infodict, ier, mesg = fsolve(myF, zGuess, full_output=True, xtol = .01)
        #print (z, " ", ier, mesg)
        
        #
        # Count the unstable samples
        # 
        if Ri < 0:
            riNegCount = riNegCount + 1
            
            #
            # Count the number of systems that converge
            #
            if ier == 1:
                riNegConv = riNegConv +1
                
        elif Ri > 0:
            #
            # Count the stable samples
            # 
            riPosCount = riPosCount + 1
            
            #
            # Count the number of stable systems that converge
            #
            if ier == 1:
                riPosConv = riPosConv +1
                #print(z[0], " ", z[1])
                
        #
        # If the system was solvable, record the solutions. They are added to our derived dataset
        #
        mostustar_col_index = mvco.columns.get_loc("MOSTustar" + mo_tag)
        l_col_index = mvco.columns.get_loc("L" + mo_tag)
        if ier == 1:
            mvco.iloc[i, mostustar_col_index] = z[0]
            mvco.iloc[i, l_col_index] = z[1]
        else:
            #
            # If the system did not converge lets try to find out why, keep the bulk Ri 
            #
            if (Ri > 0):
                posNoConvRi.append(Ri)
                #print("*******************************") 
                #print("Ri: ", Ri)
                #print("skinPotT: ", skinPotT)
                #print("T: ", T)
                #print("wspd40: ", wspd40)
                #print("waveHt: ", waveHt)
                #print("potT40: ", potT40)
                #print("Cp: ", Cp )
                #print("*******************************")
                

#
# Output convergence statistics
#
print("unstable systems: " , riNegCount, " % convergence: ", riNegConv/riNegCount)
print("stable systems: ", riPosCount, " % convergence: ", riPosConv/riPosCount)

#
# Check the bulk richardson number for oddities in convergence vs no convergence cases
#
print("ri no conv min: " , min(posNoConvRi))
print("ri no conv max: " , max(posNoConvRi))
print("Ri dataset min: ", mvco[bulk_richardson].min())
print("Ri dataset max: ", mvco[bulk_richardson].max())


In [ ]:

#
# Finally, compute temperature scale using MOST and add to the derived dataset
#
for i in range(2):
    if i == 0:
        psi_h = psi_h_branko
        psi_m = psi_m_branko
        mo_tag = '-branko'
    elif i == 1:
        psi_h = psi_h_alternate
        psi_m = psi_m_alternate
        mo_tag = '-alternate'

    mvco["TempScaleMOST" + mo_tag] = - mvco["MOSTustar" + mo_tag]  * mvco["MOSTustar" + mo_tag] * mvco['water_sfc_temperature:0_m:K']/(g*mvco["L" + mo_tag])

to resolve the bug above ^, take a look at rows 108 to 114. check the values, print them out to see if there is a nan, try to find what changed in row 112 that broke it. because it seems to have worked up until row 111, out of 400,000 rows.

In [ ]:
# view obukhov length
mvco.hist(column = 'L-branko', bins = range(-500,1500,100))
print()

In [ ]:
# view obukhov length
mvco.hist(column = 'L-alternate', bins = range(-500,1500,100))
print()

In [ ]:
print(mvco.columns)

In [ ]:
print(len(mvco))
print(mvco.isna().sum()/len(mvco))

In [ ]:
nn = mvco['L'].isna().sum()
print(nn)
print('all', all / 3 / 24 / 365)
print(nn / 3 / 24 / 365)

all = len(mvco)
print((all - nn) / 3 / 24 / 365)
# print((all - nn) / all * 100)

In [ ]:
nn = mvco['MOSTustar'].isna().sum()
print(nn)
print('all', all / 3 / 24 / 365)
print(nn / 3 / 24 / 365)

all = len(mvco)
print((all - nn) / 3 / 24 / 365)
# print((all - nn) / all * 100)

In [ ]:
def mo_fluxes_branko(u, t):
    #specific heat at constant pressure, cp=1003.5 J kg-1K-1
    ad = 1.293 # density of Pure, dry air
    # u, t = mo_similarity_offshore_branko(*args)
    mf = u**2
    cp = 1003.5
    hf = t * cp * u * -1 * ad

    return mf, hf

def mo_fluxes_alternate(u, t):
    #specific heat at constant pressure, cp=1003.5 J kg-1K-1
    ad = 1.293 # density of Pure, dry air
    # u, t = mo_similarity_offshore_alternate(*args)
    mf = u**2
    cp = 1003.5
    hf = t * cp * u * -1 * ad

    return mf, hf

In [ ]:
mvco['mf-branko'], mvco['hf-branko'] = mo_fluxes_branko(mvco['MOSTustar-branko'], mvco['L-branko'])
mvco['mf-alternate'], mvco['hf-alternate'] = mo_fluxes_branko(mvco['MOSTustar-alternate'], mvco['L-alternate'])

In [ ]:
#
# friction vel calculated by MOST (blue), trace 26 is friction vel measured (red)
#
trace30 = go.Scatter(
    x =  mvco.index,
    y =  mvco['MOSTustar'],
    name = "MOSTustar",
    line = dict(color = 'blue'),
    connectgaps = False,
    opacity = 0.5)

data = [trace30,  trace26]
plotly.offline.iplot(data, filename='basic-line-plot')
layout = timeSeriesLayout

In [ ]:
#
# We are going to use drennan surface roughness but this is the difference field
#
tracez01= go.Scatter(
    x =  mvco.index,
    y =  mvco['surface_roughness_drennan:0_m:m'] - mvco['surface_roughness_charnock:0_m:m'],
    name = "z0 drennan - charnock",
    line = dict(color = 'blue'),
    connectgaps = False,
    opacity = 0.5)



data = [tracez01]
plotly.offline.iplot(data, filename='basic-line-plot')
layout = timeSeriesLayout

In [ ]:
#
# We have added some columns with MOST calculations, lets see what we have now
#
print(mvco.columns)

In [ ]:
#
# Write the derived data with the MOST calcs to disk
#
mvco.to_csv("mvco_mlsl_qc_andMOST.csv")
